# MLOps Pipeline — Intelligent Vehicle Damage Assessment

**Full end-to-end ClearML pipeline** using pre-trained YOLO11m and YOLOv8m weights
and the CarDD dataset already versioned in the ClearML workspace.

```
Stage 0 → Install & credentials
Stage 1 → Dataset versioning  (retrieve CarDD from ClearML)
Stage 2 → Experiment tracking (register pre-trained weights + evaluate)
Stage 3 → Model registry      (side-by-side comparison & promotion)
Stage 4 → Automated pipeline  (PipelineDecorator)
Stage 5 → Performance monitoring + auto-retrain trigger
```

> **Prerequisites**  
> 1. ClearML account at https://app.clear.ml (free)  
> 2. CarDD dataset already uploaded to your ClearML workspace (Dataset ID pre-filled below)  
> 3. Pre-trained weights at `../models/yolo11m_best.pt` and `../models/yolov8m_best.pt`


---
## Stage 0 — Install & Credentials
Run once to authenticate this machine with your ClearML workspace.


In [3]:
import subprocess, sys
pkgs = ['clearml', 'ultralytics', 'pycocotools', 'tabulate']
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✓ Dependencies ready.')



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


✓ Dependencies ready.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [4]:
import clearml
print(f'ClearML version: {clearml.__version__}')
print()
print('If this is your first run, execute in a terminal:')
print('  clearml-init')
print()
print('OR paste your credentials block below (Stage 0b).')


ClearML version: 2.1.5

If this is your first run, execute in a terminal:
  clearml-init

OR paste your credentials block below (Stage 0b).


### Stage 0b — Paste credentials *(skip if `clearml-init` already run)*


In [ ]:
# ── PASTE YOUR CREDENTIALS HERE ──────────────────────────────────────────────
# Get them from: https://app.clear.ml → Settings → Workspace → Create credentials
#
# CLEARML_CREDENTIALS = """
# api {
#   web_server:   https://app.clear.ml
#   api_server:   https://api.clear.ml
#   files_server: https://files.clear.ml
#   credentials {
#     "access_key" = "PASTE_ACCESS_KEY"
#     "secret_key"  = "PASTE_SECRET_KEY"
#   }
# }
# """
#
# import os
# conf_path = os.path.expanduser('~/clearml.conf')
# with open(conf_path, 'w') as f:
#     f.write(CLEARML_CREDENTIALS)
# print(f'Credentials written to {conf_path} — restart kernel, then run Stage 1.')

print('Skipping credential paste (already configured via clearml-init).')


---
## Stage 1 — Dataset Versioning
> **HLD Stage:** Data Analysis → Feature Store

CarDD has already been uploaded to your ClearML workspace.  
This stage retrieves a local copy so every downstream cell is reproducible.


In [ ]:
from clearml import Dataset
import os
from pathlib import Path

# ── Dataset IDs already in ClearML workspace ────────────────────────────
CARDD_DATASET_ID = '0a5ce277be174c47bd75b3ef202a3b50'   # CarDD_YOLO dataset

# Fallback: use local data if ClearML dataset not reachable
LOCAL_YOLO_DATA = Path('../data/CarDD/CarDD_YOLO')
print('Retrieving CarDD dataset from ClearML...')

try:
    cardd_dataset = Dataset.get(dataset_id=CARDD_DATASET_ID)
    DATASET_PATH = Path(cardd_dataset.get_local_copy())
    SOURCE = 'ClearML'
except Exception as e:
    print(f'  ⚠ ClearML retrieval failed ({e}), falling back to local copy.')
    DATASET_PATH = LOCAL_YOLO_DATA
    SOURCE = 'local'

print(f'\n✓ Dataset ready ({SOURCE})')
print(f'  Path : {DATASET_PATH}')

# Find the dataset YAML
yaml_candidates = list(DATASET_PATH.glob('*.yaml')) + list(DATASET_PATH.glob('**/*.yaml'))
if yaml_candidates:
    DATASET_YAML = str(yaml_candidates[0])
    print(f'  YAML : {DATASET_YAML}')
else:
    DATASET_YAML = None
    print('  ⚠ No YAML found — will generate one.')


Retrieving CarDD dataset from ClearML...

✓ Dataset ready (ClearML)
  Path : /Users/bush/.clearml/cache/storage_manager/datasets/ds_0a5ce277be174c47bd75b3ef202a3b50
  YAML : /Users/bush/.clearml/cache/storage_manager/datasets/ds_0a5ce277be174c47bd75b3ef202a3b50/cardd_abs.yaml


In [9]:
import yaml

# If no YAML, generate one pointing to the local paths
if DATASET_YAML is None:
    generated_yaml = DATASET_PATH / 'cardd.yaml'
    dataset_cfg = {
        'path': str(DATASET_PATH.resolve()),
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc': 6,
        'names': ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat'],
    }
    with open(generated_yaml, 'w') as f:
        yaml.dump(dataset_cfg, f, default_flow_style=False)
    DATASET_YAML = str(generated_yaml)
    print(f'✓ Generated dataset YAML: {DATASET_YAML}')
else:
    # Patch path to be absolute (YOLO needs this)
    with open(DATASET_YAML, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['path'] = str(DATASET_PATH.resolve())
    patched_yaml = DATASET_PATH / 'cardd_abs.yaml'
    with open(patched_yaml, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)
    DATASET_YAML = str(patched_yaml)
    print(f'✓ Dataset YAML (patched with absolute path): {DATASET_YAML}')

# Count images
for split in ['train', 'val', 'test']:
    split_dir = DATASET_PATH / 'images' / split
    if split_dir.exists():
        n = len(list(split_dir.glob('*')))
        print(f'  {split:6s}: {n} images')


✓ Dataset YAML (patched with absolute path): /Users/bush/.clearml/cache/storage_manager/datasets/ds_0a5ce277be174c47bd75b3ef202a3b50/cardd_abs.yaml
  train : 2816 images
  val   : 810 images
  test  : 374 images


### Stage 1b — Create a New Dataset Version *(optional)*
Run this to snapshot the current local CarDD_YOLO folder as a new ClearML dataset version.
Safe to skip if you only want to consume the existing dataset.


In [10]:
# ── Uncomment and run to create a new versioned snapshot ─────────────────────

# new_ds = Dataset.create(
#     dataset_name='CarDD_YOLO',
#     dataset_project='Intelligent Vehicle Damage Assessment',
# )
# new_ds.add_files(str(LOCAL_YOLO_DATA))
# new_ds.upload()
# new_ds.finalize()
# print(f'✓ New dataset version created: {new_ds.id}')

print('Stage 1b skipped (using existing dataset ID).')


Stage 1b skipped (using existing dataset ID).


---
### Stage 1c — Update Dataset Version
> Run this whenever uploading a **newer version** of CarDD to ClearML.
> It creates a new immutable dataset version that all subsequent stages reference.

**Workflow:**
1. Prepare your updated `CarDD_YOLO` folder locally (`NEW_DATA_PATH`)
2. Run the three cells below — a new ClearML dataset version is created
3. `UPDATED_DATASET_ID` is set automatically — Stage 2b and Stage 4 pick it up


In [11]:
# ── Configure new data location ──────────────────────────────────────────────
from pathlib import Path
import datetime

# Point this to whatever folder holds your new CarDD_YOLO data
NEW_DATA_PATH = Path('../data/CarDD/CarDD_YOLO')  # e.g. Path('/tmp/CarDD_YOLO_v2')

# Parent dataset — new version inherits from this (only changed files are stored)
PARENT_DATASET_ID = '0a5ce277be174c47bd75b3ef202a3b50'  # original CarDD_YOLO

VERSION_TAG = datetime.datetime.now().strftime('%Y%m%d_%H%M')

print(f'New data path : {NEW_DATA_PATH.resolve()}')
print(f'Parent ID     : {PARENT_DATASET_ID}')
print(f'Version tag   : {VERSION_TAG}')
print(f'Path exists   : {NEW_DATA_PATH.exists()}')


New data path : /Users/bush/Library/CloudStorage/OneDrive-UTS/UTS Master/42028 Deep Learning and Convolutional Neural Network/Assignment 3/vehicle_damage_assessment/data/CarDD/CarDD_YOLO
Parent ID     : 0a5ce277be174c47bd75b3ef202a3b50
Version tag   : 20260409_1701
Path exists   : True


In [ ]:
from clearml import Dataset
import yaml

# ── Create child version  ───────────────────────
print(f'Creating new dataset version: CarDD_YOLO v{VERSION_TAG}...')

new_ds = Dataset.create(
    dataset_name    = 'CarDD_YOLO',
    dataset_project = 'Intelligent Vehicle Damage Assessment',
    dataset_version = VERSION_TAG,
    parent_datasets = [PARENT_DATASET_ID] if PARENT_DATASET_ID else [],
)

print(f'  Adding files from: {NEW_DATA_PATH.resolve()}')
new_ds.add_files(str(NEW_DATA_PATH.resolve()))

print('  Uploading...')
new_ds.upload(output_url=None)  # None -> default ClearML storage
new_ds.finalize()               # make immutable

UPDATED_DATASET_ID = new_ds.id
print(f'\n  ID      : {UPDATED_DATASET_ID}')
print(f'  View at : https://app.clear.ml -> Datasets -> CarDD_YOLO')
print('  Done!')


In [ ]:
# ── Retrieve local copy & patch YAML ─────────────────────────────────────────
import yaml
from clearml import Dataset
from pathlib import Path

updated_ds      = Dataset.get(dataset_id=UPDATED_DATASET_ID)
UPDATED_DS_PATH = Path(updated_ds.get_local_copy())

updated_yaml_path = UPDATED_DS_PATH / 'cardd_updated.yaml'
updated_cfg = {
    'path':  str(UPDATED_DS_PATH.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    6,
    'names': ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat'],
}
with open(updated_yaml_path, 'w') as yf:
    yaml.dump(updated_cfg, yf, default_flow_style=False)

UPDATED_DATASET_YAML = str(updated_yaml_path)

print(f'Dataset path : {UPDATED_DS_PATH}')
print(f'Dataset YAML : {UPDATED_DATASET_YAML}')
for split in ['train', 'val', 'test']:
    d = UPDATED_DS_PATH / 'images' / split
    n = len(list(d.glob('*'))) if d.exists() else 0
    print(f'  {split:6s} : {n} images')
print('Stage 1c complete. Now run Stage 2b to train on the new data.')


---
## Stage 2 — Experiment Tracking
> **HLD Stage:** Orchestrated Experiment → Source Code → Source Repository

Instead of re-training from scratch, we **register the already-trained weights**
(`yolo11m_best.pt` and `yolov8m_best.pt`) as ClearML tasks, log all final
metrics, and link the CarDD dataset.  This lets the UI show a complete audit
trail without burning compute.


In [2]:
# ── Global config ───────────────────────────────────────
import os
from pathlib import Path

import torch

def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

DEVICE = get_device()
print(f'\n[INFO] Auto-detected execution device: {DEVICE.upper()}')

PROJECT_NAME   = 'Intelligent Vehicle Damage Assessment'
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45

# Trained weight paths (relative to this notebook)
WEIGHTS = {
    'YOLO11m': Path('../models/yolo11m_best.pt').resolve(),
    'YOLOv8m': Path('../models/yolov8m_best.pt').resolve(),
}

CLASS_NAMES = ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']
LABEL_ENUM  = {name: idx for idx, name in enumerate(CLASS_NAMES)}

# Verify weights exist
for name, path in WEIGHTS.items():
    status = '✓' if path.exists() else '✗ MISSING'
    print(f'  {status}  {name}: {path}')



[INFO] Auto-detected execution device: MPS
  ✓  YOLO11m: /Users/bush/Library/CloudStorage/OneDrive-UTS/UTS Master/42028 Deep Learning and Convolutional Neural Network/Assignment 3/vehicle_damage_assessment/models/yolo11m_best.pt
  ✓  YOLOv8m: /Users/bush/Library/CloudStorage/OneDrive-UTS/UTS Master/42028 Deep Learning and Convolutional Neural Network/Assignment 3/vehicle_damage_assessment/models/yolov8m_best.pt


In [3]:
from clearml import Task, Dataset, OutputModel
from ultralytics import YOLO

# Store task IDs so Stage 3 can consume them
registered_task_ids = {}

for model_name, weights_path in WEIGHTS.items():
    if not weights_path.exists():
        print(f'⚠ Skipping {model_name} — weights not found at {weights_path}')
        continue

    print(f'\n{'─'*60}')
    print(f'Registering {model_name}...')

    # ── Init task ────────────────────────────────────────────────────────────
    task = Task.init(
        project_name=PROJECT_NAME,
        task_name=f'{model_name}-PreTrained-Evaluation',
        task_type=Task.TaskTypes.testing,
        reuse_last_task_id=False,
    )

    # ── Connect hyperparameters ───────────────────────────────────────────────
    params = task.connect({
        'model_name':      model_name,
        'weights_path':    str(weights_path),
        'dataset_id':      CARDD_DATASET_ID,
        'conf_threshold':  CONF_THRESHOLD,
        'iou_threshold':   IOU_THRESHOLD,
        'classes':         CLASS_NAMES,
    })

    # ── Link dataset ──────────────────────────────────────────────────────────
    task.connect_configuration(DATASET_YAML, name='dataset_config')
    task.connect_configuration('../configs/training_config.yaml', name='training_config')

    # ── Validate model on CarDD test set ─────────────────────────────────────
    print(f'  Running validation on CarDD test split...')
    yolo = YOLO(str(weights_path))
    val_metrics = yolo.val(
        data=DATASET_YAML,
        split='test',
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False,
        device=DEVICE,
    )

    metrics_dict = {
        'mAP@0.5':    round(float(val_metrics.box.map50),  4),
        'mAP@0.5:95': round(float(val_metrics.box.map),    4),
        'Precision':  round(float(val_metrics.box.mp),     4),
        'Recall':     round(float(val_metrics.box.mr),     4),
        'F1':         round(2 * float(val_metrics.box.mp) * float(val_metrics.box.mr) /
                           max(float(val_metrics.box.mp) + float(val_metrics.box.mr), 1e-9), 4),
    }

    # ── Log scalars ───────────────────────────────────────────────────────────
    logger = task.get_logger()
    for metric_name, value in metrics_dict.items():
        logger.report_scalar('Final Metrics', metric_name, value=value, iteration=0)
        print(f'    {metric_name}: {value}')

    # ── Log per-class metrics ─────────────────────────────────────────────────
    if hasattr(val_metrics.box, 'ap_class_index'):
        for ci, cls_idx in enumerate(val_metrics.box.ap_class_index):
            cls_name = CLASS_NAMES[cls_idx] if cls_idx < len(CLASS_NAMES) else f'cls_{cls_idx}'
            ap50 = float(val_metrics.box.ap50[ci]) if hasattr(val_metrics.box, 'ap50') else 0
            logger.report_scalar('Per-Class AP@0.5', cls_name, value=ap50, iteration=0)

    # ── Register model weights in ClearML Model Registry ─────────────────────
    output_model = OutputModel(
        task=task,
        label_enumeration=LABEL_ENUM,
        name=f'{model_name}-CarDD',
    )
    output_model.update_weights(str(weights_path), auto_delete_file=False)
    output_model.update_design(config_dict={'metrics': metrics_dict, **params})

    task.close()
    registered_task_ids[model_name] = task.id
    print(f'  ✓ Task ID saved: {task.id}')

print(f'\n✓ All models registered!')
print(f'  Task IDs: {registered_task_ids}')



────────────────────────────────────────────────────────────
Registering YOLO11m...
ClearML Task: created new task id=0a1e4abaa73746adb442be904c04344b
ClearML results page: https://app.clear.ml/projects/f9f612bc132c4e8689723b061db5425e/experiments/0a1e4abaa73746adb442be904c04344b/output/log
2026-04-09 17:02:41,300 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
  Running validation on CarDD test split...
2026-04-09 17:02:54,282 - clearml.model - WARNING - 2 model found when searching for `file:///Users/bush/Library/CloudStorage/OneDrive-UTS/UTS Master/42028 Deep Learning and Convolutional Neural Network/Assignment 3/vehicle_damage_assessment/models/yolo11m_best.pt`
2026-04-09 17:02:54,283 - clearml.model - WARNING - Selected model `Imported by Smoke-YOLO11m-Eval` (id=c0b4a37da64c45a58c467e2ab88fbcd9)
Ultralytics 8.4.33 🚀 Python-3.14.3 torch-2.11.0 MP

Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/f9f612bc132c4e8689723b061db5425e/experiments/66974d09587442e8a97a2ead6c073dbe/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
  Running validation on CarDD test split...
Ultralytics 8.4.33 🚀 Python-3.14.3 torch-2.11.0 MPS (Apple M5 Pro)
Model summary (fused): 93 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2579.8±701.1 MB/s, size: 801.8 KB)
val: Scanning /Users/bush/.clearml/cache/storage_manager/datasets/ds_0a5ce277be174c47bd75b3ef202a3b50/labels/test.cache... 374 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 374/374 120.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.5it/s 9.8s0.4s
                   all        374        785      0.741      0.702       0.75      0.608
Speed: 0.1ms preprocess, 8.1ms inference, 0.0ms loss, 5.1ms postproce

---
### Stage 2b — Full Training Task
> Use after Stage 1c when you have a new dataset version and want to
> **fine-tune YOLO11m or YOLOv8m** on the updated CarDD data.

ClearML auto-captures:
- All hyperparameters (editable in the UI before queuing on a remote agent)
- Loss curves, mAP, precision, recall **per epoch**
- GPU utilisation and system metrics
- The best `.pt` weights registered in the Model Registry


In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# Set model_name to 'yolo11m' or 'yolov8m'

TRAIN_CONFIG = {
    'model_name'    : 'yolo11m',
    'base_weights'  : '../notebooks/yolo11m.pt',   # backbone to fine-tune from
    'epochs'        : 50,
    'batch_size'    : 16,
    'image_size'    : 640,
    'lr0'           : 0.001,
    'lrf'           : 0.01,
    'weight_decay'  : 0.0005,
    'conf_threshold': 0.25,
    'iou_threshold' : 0.45,
    'device'        : DEVICE,
    'dataset_id'    : None,   # auto-filled below
    'project'       : 'runs/clearml_train',
}

# Auto-select dataset: prefer updated version (Stage 1c), fall back to original
if 'UPDATED_DATASET_ID' in dir() and UPDATED_DATASET_ID:
    TRAIN_CONFIG['dataset_id'] = UPDATED_DATASET_ID
    _train_ds_yaml = UPDATED_DATASET_YAML
    print(f'Dataset : UPDATED ({UPDATED_DATASET_ID})')
else:
    TRAIN_CONFIG['dataset_id'] = CARDD_DATASET_ID
    _train_ds_yaml = DATASET_YAML
    print(f'Dataset : ORIGINAL ({CARDD_DATASET_ID})')

from pathlib import Path
bw = Path(TRAIN_CONFIG['base_weights'])
print(f'Model   : {TRAIN_CONFIG["model_name"]}')
print(f'Weights : {bw.resolve()}  {"found" if bw.exists() else "MISSING"}')
print(f'Epochs  : {TRAIN_CONFIG["epochs"]}')


In [ ]:
from clearml import Task, Dataset, OutputModel
from ultralytics import YOLO
from pathlib import Path
import yaml

CLASS_NAMES = ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']
LABEL_ENUM  = {n: i for i, n in enumerate(CLASS_NAMES)}
MODEL_NAME  = TRAIN_CONFIG['model_name']

_vtag = VERSION_TAG if 'VERSION_TAG' in dir() else 'manual'

# ── Init task ─────────────────────────────────────────────────────────────────
train_task = Task.init(
    project_name       = PROJECT_NAME,
    task_name          = f'{MODEL_NAME}-Training-v{_vtag}',
    task_type          = Task.TaskTypes.training,
    reuse_last_task_id = False,
)

# Connect hyperparameters — editable from ClearML UI before remote queuing
params = train_task.connect(TRAIN_CONFIG, name='Training')
train_task.connect_configuration('../configs/training_config.yaml', name='training_config')
train_task.connect_configuration(_train_ds_yaml, name='dataset_config')

print(f'ClearML task : {train_task.name}  (ID: {train_task.id})')

# ── Load versioned dataset ────────────────────────────────────────────────────
ds       = Dataset.get(dataset_id=params['dataset_id'])
ds_path  = Path(ds.get_local_copy())
run_yaml = ds_path / f'cardd_{MODEL_NAME}_run.yaml'
run_cfg  = {
    'path':  str(ds_path.resolve()),
    'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
    'nc': 6, 'names': CLASS_NAMES,
}
with open(run_yaml, 'w') as ryf:
    yaml.dump(run_cfg, ryf, default_flow_style=False)
print(f'Dataset path : {ds_path}')

# ── Train ─────────────────────────────────────────────────────────────────────
print(f'\nStarting training...')
model = YOLO(str(Path(params['base_weights']).resolve()))
results = model.train(
    data         = str(run_yaml),
    epochs       = params['epochs'],
    batch        = params['batch_size'],
    imgsz        = params['image_size'],
    lr0          = params['lr0'],
    lrf          = params['lrf'],
    weight_decay = params['weight_decay'],
    project      = params['project'],
    name         = MODEL_NAME,
    exist_ok     = True,
    device       = params.get('device', 'cpu'),
)
print('Training complete!')


In [ ]:
# ── Log final metrics & register weights ─────────────────────────────────────
train_logger = train_task.get_logger()

rd = results.results_dict
precision = rd.get('metrics/precision', 0)
recall    = rd.get('metrics/recall',    0)
final_metrics = {
    'mAP@0.5'    : rd.get('metrics/mAP50',    0),
    'mAP@0.5:95' : rd.get('metrics/mAP50-95', 0),
    'Precision'  : precision,
    'Recall'     : recall,
    'F1'         : round(2 * precision * recall / max(precision + recall, 1e-9), 4),
}

print('Final metrics:')
for k, v in final_metrics.items():
    train_logger.report_scalar('Final Metrics', k, value=float(v), iteration=0)
    print(f'  {k}: {float(v):.4f}')

# Locate best.pt
best_pt = Path(params['project']) / MODEL_NAME / 'weights' / 'best.pt'
if not best_pt.exists():
    cands = list(Path('runs').rglob(f'{MODEL_NAME}/weights/best.pt'))
    best_pt = cands[0] if cands else best_pt

# Register in ClearML Model Registry
output_model = OutputModel(
    task=train_task,
    label_enumeration=LABEL_ENUM,
    name=f'{MODEL_NAME}-CarDD-Trained',
)
output_model.update_weights(str(best_pt.resolve()), auto_delete_file=False)
output_model.update_design(config_dict={**params, **{k: float(v) for k, v in final_metrics.items()}})

TRAINED_TASK_ID = train_task.id
train_task.close()

print(f'\nModel registered in ClearML Model Registry')
print(f'  Task ID : {TRAINED_TASK_ID}  <- add to Stage 3 MODELS_TO_COMPARE')
print(f'  Weights : {best_pt.resolve()}')
print(f'  View at : https://app.clear.ml -> {PROJECT_NAME}')
print('\nTo train the other model, change model_name in TRAIN_CONFIG and re-run Stage 2b.')


---
## Stage 3 — Model Evaluation & Registry
> **HLD Stage:** Model Evaluation → Model Validation → Model Registry

Compare YOLO11m vs YOLOv8m side-by-side, log a comparison table to ClearML
and automatically promote the winner to the Model Registry.


In [4]:
from clearml import Task, Model
from ultralytics import YOLO
from tabulate import tabulate
import json

# ── Use task IDs from Stage 2 (auto-filled) ──────────────────────────────────
# If you re-run Stage 3 standalone, paste your task IDs here:
MODELS_TO_COMPARE = registered_task_ids.copy()  # {'YOLO11m': 'abc123', 'YOLOv8m': 'def456'}

# Fallback: run validation locally if task IDs are not available
if not MODELS_TO_COMPARE:
    print('No task IDs from Stage 2 — running local validation.')
    MODELS_TO_COMPARE = {name: None for name in WEIGHTS}

print(f'Comparing models: {list(MODELS_TO_COMPARE.keys())}')


Comparing models: ['YOLO11m', 'YOLOv8m']


In [5]:
eval_task = Task.init(
    project_name=PROJECT_NAME,
    task_name='Model-Evaluation-Comparison',
    task_type=Task.TaskTypes.testing,
    reuse_last_task_id=False,
)
eval_logger = eval_task.get_logger()

comparison = {}   # model_name → metrics dict
inference_times = {}

for model_name, task_id in MODELS_TO_COMPARE.items():
    print(f'\nEvaluating {model_name}...')

    # Get weights: either from ClearML registry or local path
    if task_id:
        try:
            src_task = Task.get_task(task_id=task_id)
            registered_model = Model(task=src_task)
            weights_path = registered_model.get_local_copy()
        except Exception as e:
            print(f'  ⚠ Could not fetch from registry ({e}), using local weights.')
            weights_path = str(WEIGHTS[model_name])
    else:
        weights_path = str(WEIGHTS[model_name])

    # Run validation
    import time
    yolo = YOLO(weights_path)
    t0 = time.time()
    val_metrics = yolo.val(
        data=DATASET_YAML,
        split='test',
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False,
        device=DEVICE,
    )
    infer_ms = (time.time() - t0) * 1000 / max(len(list((DATASET_PATH / 'images' / 'test').glob('*'))), 1)

    m = {
        'mAP@0.5':    round(float(val_metrics.box.map50), 4),
        'mAP@0.5:95': round(float(val_metrics.box.map),   4),
        'Precision':  round(float(val_metrics.box.mp),    4),
        'Recall':     round(float(val_metrics.box.mr),    4),
        'F1':         round(2 * float(val_metrics.box.mp) * float(val_metrics.box.mr) /
                           max(float(val_metrics.box.mp) + float(val_metrics.box.mr), 1e-9), 4),
        'Inference_ms': round(infer_ms, 2),
    }
    comparison[model_name] = m

    # Log scalars to ClearML
    for metric, value in m.items():
        eval_logger.report_scalar(f'Comparison/{metric}', model_name, value, 0)
    print(f'  mAP@0.5={m["mAP@0.5"]}  mAP@0.5:95={m["mAP@0.5:95"]}  F1={m["F1"]}')

# ── Comparison table ─────────────────────────────────────────────────────────
import pandas as pd
rows = [{'Model': k, **v} for k, v in comparison.items()]
df = pd.DataFrame(rows)
print('\n' + '='*70)
print('MODEL COMPARISON — CarDD Test Set')
print('='*70)
print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))

# Log table to ClearML
eval_logger.report_table('Model Comparison', 'CarDD Test Results', iteration=0, table_plot=df)


ClearML Task: created new task id=d48c52fbb70e482f8289bebec15f7527


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/f9f612bc132c4e8689723b061db5425e/experiments/d48c52fbb70e482f8289bebec15f7527/output/log

Evaluating YOLO11m...
  ⚠ Could not fetch from registry (Model.__init__() got an unexpected keyword argument 'task'), using local weights.
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


Connecting multiple input models with the same name: `yolo11m_best`. This might result in the wrong model being used when executing remotely


Ultralytics 8.4.33 🚀 Python-3.14.3 torch-2.11.0 MPS (Apple M5 Pro)
YOLO11m summary (fused): 126 layers, 20,034,658 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.2±0.1 ms, read: 2538.4±484.6 MB/s, size: 801.8 KB)
val: Scanning /Users/bush/.clearml/cache/storage_manager/datasets/ds_0a5ce277be174c47bd75b3ef202a3b50/labels/test.cache... 374 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 374/374 92.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.3it/s 10.6s0.5s
                   all        374        785       0.81      0.704      0.773      0.621
Speed: 0.2ms preprocess, 10.7ms inference, 0.0ms loss, 4.9ms postprocess per image
Results saved to /Users/bush/Library/CloudStorage/OneDrive-UTS/UTS Master/42028 Deep Learning and Convolutional Neural Network/Assignment 3/vehicle_damage_assessment/notebooks/runs/detect/val4
  mAP@0.5=0.7727  mAP@0.5:95=0.6215  F1=0.7534

Evaluating YOLOv

Connecting multiple input models with the same name: `yolov8m_best`. This might result in the wrong model being used when executing remotely


Ultralytics 8.4.33 🚀 Python-3.14.3 torch-2.11.0 MPS (Apple M5 Pro)
Model summary (fused): 93 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 545.8±74.8 MB/s, size: 611.8 KB)
val: Scanning /Users/bush/.clearml/cache/storage_manager/datasets/ds_0a5ce277be174c47bd75b3ef202a3b50/labels/test.cache... 374 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 374/374 120.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.5it/s 9.5s0.4s
                   all        374        785      0.741      0.702       0.75      0.608
Speed: 0.2ms preprocess, 7.7ms inference, 0.0ms loss, 4.8ms postprocess per image
Results saved to /Users/bush/Library/CloudStorage/OneDrive-UTS/UTS Master/42028 Deep Learning and Convolutional Neural Network/Assignment 3/vehicle_damage_assessment/notebooks/runs/detect/val5
  mAP@0.5=0.7499  mAP@0.5:95=0.6085  F1=0.721

MODEL COMPARISON — CarD

In [6]:
# ── Select Best Model ─────────────────────────────────────────────────────────
best_model_name = max(comparison, key=lambda m: comparison[m]['mAP@0.5:95'])
best_metrics    = comparison[best_model_name]

print(f'\n✓ Best model: {best_model_name}')
for k, v in best_metrics.items():
    print(f'  {k}: {v}')

# ── Tag as production in ClearML ──────────────────────────────────────────────
if best_model_name in MODELS_TO_COMPARE and MODELS_TO_COMPARE[best_model_name]:
    best_task = Task.get_task(task_id=MODELS_TO_COMPARE[best_model_name])
    best_task.add_tags(['production-candidate', f'best-mAP50-{best_metrics["mAP@0.5"]}'])
    print(f'  ✓ Tagged task {best_task.id} as production-candidate')

# ── Log promotion decision ────────────────────────────────────────────────────
eval_logger.report_text(
    f'Best model: {best_model_name}\n'
    f'mAP@0.5: {best_metrics["mAP@0.5"]}\n'
    f'mAP@0.5:95: {best_metrics["mAP@0.5:95"]}\n'
    f'F1: {best_metrics["F1"]}\n'
)
eval_task.close()
print('\n✓ Stage 3 complete — model promoted to registry.')



✓ Best model: YOLO11m
  mAP@0.5: 0.7727
  mAP@0.5:95: 0.6215
  Precision: 0.8104
  Recall: 0.7039
  F1: 0.7534
  Inference_ms: 32.52
  ✓ Tagged task 0a1e4abaa73746adb442be904c04344b as production-candidate
Best model: YOLO11m
mAP@0.5: 0.7727
mAP@0.5:95: 0.6215
F1: 0.7534


✓ Stage 3 complete — model promoted to registry.


---
## Stage 4 — Automated End-to-End Pipeline
> **HLD Stage:** Automated Pipeline (Data → Validation → Preparation → Training → Evaluation → Validation)

The `PipelineDecorator` wires all steps into a single trackable pipeline.  
Call `PipelineDecorator.run_locally()` for local testing; remove it to run
on a ClearML agent for full distributed execution.


In [ ]:
from clearml.automation.controller import PipelineDecorator

# ─────────────────────────────────────────────────────────────────────────────
# Step 1 — Validate & retrieve dataset
# ─────────────────────────────────────────────────────────────────────────────
@PipelineDecorator.component(
    return_values=['dataset_path', 'dataset_yaml'],
    cache=True,
    packages=['clearml', 'pyyaml'],
)
def step_retrieve_dataset(dataset_id: str, local_fallback: str) -> tuple:
    from clearml import Dataset
    from pathlib import Path
    import yaml, os

    try:
        ds = Dataset.get(dataset_id=dataset_id)
        ds_path = Path(ds.get_local_copy())
        source = 'ClearML'
    except Exception as e:
        print(f'Fallback to local ({e})')
        ds_path = Path(local_fallback)
        source = 'local'

    assert ds_path.exists(), f'Dataset path not found: {ds_path}'

    # Verify splits
    for split in ['train', 'val', 'test']:
        split_dir = ds_path / 'images' / split
        assert split_dir.exists(), f'Missing split: {split_dir}'
        n = len(list(split_dir.glob('*')))
        print(f'  {split}: {n} images')

    # Build absolute YAML
    yaml_path = ds_path / 'cardd_pipeline.yaml'
    cfg = {
        'path':  str(ds_path.resolve()),
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc':    6,
        'names': ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat'],
    }
    with open(yaml_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    print(f'✓ Dataset ready ({source}): {ds_path}')
    return str(ds_path), str(yaml_path)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 2 — Evaluate pre-trained model on CarDD
# ─────────────────────────────────────────────────────────────────────────────
@PipelineDecorator.component(
    return_values=['task_id', 'metrics'],
    cache=False,
    packages=['clearml', 'ultralytics'],
)
def step_evaluate_model(
    model_name: str,
    weights_path: str,
    dataset_yaml: str,
    conf: float,
    iou: float,
    device: str,
) -> tuple:
    from clearml import Task, OutputModel
    from ultralytics import YOLO

    task = Task.init(
        project_name='Intelligent Vehicle Damage Assessment',
        task_name=f'{model_name}-Pipeline-Evaluation',
        task_type=Task.TaskTypes.testing,
        reuse_last_task_id=False,
    )
    task.connect({
        'model_name': model_name,
        'weights_path': weights_path,
        'conf': conf,
        'iou': iou,
        'device': device,
    })

    yolo = YOLO(weights_path)
    val = yolo.val(data=dataset_yaml, split='test', conf=conf, iou=iou, verbose=False, device=device)

    CLASS_NAMES = ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']
    metrics = {
        'mAP50':    round(float(val.box.map50), 4),
        'mAP50_95': round(float(val.box.map),   4),
        'precision': round(float(val.box.mp),   4),
        'recall':   round(float(val.box.mr),    4),
        'f1': round(2 * float(val.box.mp) * float(val.box.mr) /
                   max(float(val.box.mp) + float(val.box.mr), 1e-9), 4),
    }

    logger = task.get_logger()
    for k, v in metrics.items():
        logger.report_scalar('Metrics', k, value=v, iteration=0)

    # Register weights
    om = OutputModel(
        task=task,
        label_enumeration={n: i for i, n in enumerate(CLASS_NAMES)},
        name=f'{model_name}-Pipeline',
    )
    om.update_weights(weights_path, auto_delete_file=False)

    task.close()
    print(f'✓ {model_name}: mAP50={metrics["mAP50"]}, F1={metrics["f1"]}')
    return task.id, metrics


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3 — Compare & promote best model
# ─────────────────────────────────────────────────────────────────────────────
@PipelineDecorator.component(
    return_values=['best_model', 'best_map50', 'passed'],
    cache=False,
    packages=['clearml'],
)
def step_promote_best(
    yolo11m_task_id: str, yolo11m_metrics: dict,
    yolov8m_task_id: str, yolov8m_metrics: dict,
    min_map: float,
) -> tuple:
    from clearml import Task

    candidates = {
        'YOLO11m': (yolo11m_task_id, yolo11m_metrics),
        'YOLOv8m': (yolov8m_task_id, yolov8m_metrics),
    }

    best_name, (best_id, best_m) = max(
        candidates.items(), key=lambda kv: kv[1][1]['mAP50_95']
    )

    print(f'\n{'─'*50}')
    print(f'Best model: {best_name}')
    for k, v in best_m.items():
        print(f'  {k}: {v}')

    passed = best_m['mAP50'] >= min_map
    status = 'PASS ✓' if passed else f'FAIL ✗ (mAP50={best_m["mAP50"]} < {min_map})'
    print(f'Quality gate: {status}')

    if best_id:
        t = Task.get_task(task_id=best_id)
        t.add_tags(['production-candidate', 'pipeline-promoted'])

    return best_name, best_m['mAP50'], passed


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Pipeline Controller
# ─────────────────────────────────────────────────────────────────────────────
@PipelineDecorator.pipeline(
    name='Vehicle-Damage-Assessment-Pipeline',
    project='Intelligent Vehicle Damage Assessment',
    version='2.0',
)
def run_pipeline(
    dataset_id:      str   = '0a5ce277be174c47bd75b3ef202a3b50',
    yolo11m_weights: str   = '../models/yolo11m_best.pt',
    yolov8m_weights: str   = '../models/yolov8m_best.pt',
    local_fallback:  str   = '../data/CarDD/CarDD_YOLO',
    conf:            float = 0.25,
    iou:             float = 0.45,
    min_map:         float = 0.50,
):
    from pathlib import Path
    # Resolve relative paths to absolute (pipeline steps run in different cwd)
    yolo11m_abs = str(Path(yolo11m_weights).resolve())
    yolov8m_abs = str(Path(yolov8m_weights).resolve())
    fallback_abs = str(Path(local_fallback).resolve())

    # Step 1: Retrieve & validate dataset
    ds_path, ds_yaml = step_retrieve_dataset(dataset_id, fallback_abs)

    # Step 2a: Evaluate YOLO11m
    task_id_11m, metrics_11m = step_evaluate_model(
        'YOLO11m', yolo11m_abs, ds_yaml, conf, iou, device
    )
    # Step 2b: Evaluate YOLOv8m
    task_id_v8m, metrics_v8m = step_evaluate_model(
        'YOLOv8m', yolov8m_abs, ds_yaml, conf, iou, device
    )

    # Step 3: Compare and promote
    best_model, best_map50, passed = step_promote_best(
        task_id_11m, metrics_11m,
        task_id_v8m, metrics_v8m,
        min_map,
    )

    if passed:
        print(f'\n✓ Pipeline complete. {best_model} promoted. mAP@0.5={best_map50:.4f}')
    else:
        print(f'\n✗ No model met the quality gate (min_map={min_map}). Review results.')


In [ ]:
# ── Execute Pipeline ──────────────────────────────────────────────────────────
# run_locally() → runs all steps in-process (no ClearML agent needed)
# Remove it to dispatch steps to a remote ClearML agent.

PipelineDecorator.run_locally()

run_pipeline(
    dataset_id      = '0a5ce277be174c47bd75b3ef202a3b50',
    yolo11m_weights = '../models/yolo11m_best.pt',
    yolov8m_weights = '../models/yolov8m_best.pt',
    local_fallback  = '../data/CarDD/CarDD_YOLO',
    conf            = 0.25,
    iou             = 0.45,
    min_map         = 0.50,
    device          = DEVICE,
)
print('\n✓ Pipeline execution finished.')


---
## Stage 5 — Performance Monitoring & Auto-Retrain Trigger
> **HLD Stage:** Performance Monitoring → Trigger

### 5a — Add monitoring to the FastAPI backend
The cell below shows exactly what to add to `backend/app/routers/damage.py`.


In [ ]:
# ── Code snippet to add to backend/app/routers/damage.py ─────────────────────
# This cell is for REFERENCE ONLY — do not run directly in the notebook.

MONITORING_CODE = '''
# ── At top of backend/app/routers/damage.py ──────────────────────────────────
from clearml import Task
import time

_monitor_task  = None
_request_count = 0

def _get_monitor():
    """Lazy-init a persistent ClearML monitoring task."""
    global _monitor_task
    if _monitor_task is None:
        _monitor_task = Task.init(
            project_name="Intelligent Vehicle Damage Assessment",
            task_name="Production-Monitoring",
            task_type=Task.TaskTypes.monitor,
            reuse_last_task_id=True,   # Accumulate across restarts
        )
    return _monitor_task

# ── Inside your /damage/predict endpoint, after inference: ───────────────────
global _request_count
_request_count += 1
logger = _get_monitor().get_logger()

avg_conf = sum(d.confidence for d in result.detections) / max(len(result.detections), 1)
logger.report_scalar("Production", "inference_ms",    result.inference_time_ms, _request_count)
logger.report_scalar("Production", "num_detections",  len(result.detections),   _request_count)
logger.report_scalar("Production", "avg_confidence",  avg_conf,                 _request_count)
# ─────────────────────────────────────────────────────────────────────────────
'''

print('─── Code to add to backend/app/routers/damage.py ───')
print(MONITORING_CODE)


### 5b — Simulate production inference and log metrics
This cell validates the monitoring setup by running the best model on a handful
of real CarDD test images and logging them as if they were production requests.


In [ ]:
from clearml import Task
from ultralytics import YOLO
from pathlib import Path
import time, random

MONITOR_PROJECT = 'Intelligent Vehicle Damage Assessment'

print('Starting production monitoring simulation...')

monitor_task = Task.init(
    project_name=MONITOR_PROJECT,
    task_name='Production-Monitoring-Simulation',
    task_type=Task.TaskTypes.monitor,
    reuse_last_task_id=False,
)
mon_logger = monitor_task.get_logger()

# Load best model weights
best_weights = WEIGHTS.get(best_model_name, list(WEIGHTS.values())[0])
yolo = YOLO(str(best_weights))

# Sample up to 20 test images
test_dir = DATASET_PATH / 'images' / 'test'
img_files = list(test_dir.glob('*.jpg')) + list(test_dir.glob('*.png'))
sample_imgs = random.sample(img_files, min(20, len(img_files)))

print(f'  Simulating {len(sample_imgs)} requests with {best_model_name}...')

for req_idx, img_path in enumerate(sample_imgs, start=1):
    t0 = time.time()
    results = yolo.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)
    infer_ms = (time.time() - t0) * 1000

    boxes = results[0].boxes
    n_det = len(boxes) if boxes is not None else 0
    avg_conf = float(boxes.conf.mean()) if n_det > 0 else 0.0

    mon_logger.report_scalar('Production', 'inference_ms',   infer_ms,  req_idx)
    mon_logger.report_scalar('Production', 'num_detections', n_det,     req_idx)
    mon_logger.report_scalar('Production', 'avg_confidence', avg_conf,  req_idx)

    if req_idx % 5 == 0:
        print(f'  [{req_idx:2d}] infer={infer_ms:.1f}ms  det={n_det}  conf={avg_conf:.3f}')

monitor_task.close()
print(f'\n✓ Monitoring simulation complete — {len(sample_imgs)} requests logged.')
print(f'  View at: https://app.clear.ml → {MONITOR_PROJECT} → Production-Monitoring-Simulation')


### 5c — Auto-Retrain Trigger
The scheduler watches `Production/avg_confidence` and re-runs the full pipeline
when confidence drops by more than 5%.


In [ ]:
# ── Fill in IDs after Stage 4 and 5b have run at least once ─────────────────
# MONITORING_TASK_ID = 'PASTE_MONITORING_TASK_ID_HERE'
# PIPELINE_TASK_ID   = 'PASTE_PIPELINE_TASK_ID_HERE'

# Example auto-retrain trigger (uncomment when IDs are available):
# -------------------------------------------------------------------
# from clearml.automation import TriggerScheduler
#
# scheduler = TriggerScheduler()
# scheduler.add_task_trigger(
#     schedule_task_id   = MONITORING_TASK_ID,
#     trigger_on_metric  = 'Production/avg_confidence',
#     trigger_on_sign    = -1,        # fires when metric DECREASES
#     trigger_threshold  = 0.05,      # by > 5%
#     schedule_pipeline_id = PIPELINE_TASK_ID,
# )
# scheduler.start()  # blocking; run in a dedicated process
# -------------------------------------------------------------------

print('✓ Trigger configuration ready (see comments above).')
print('  Once you have task IDs from Stage 4 and 5b, uncomment and run.')


---
## Summary

| Stage | HLD Component | ClearML Feature | Status |
|-------|--------------|-----------------|--------|
| 0 | — | `clearml-init` credentials | Run once |
| 1 | Feature Store | `Dataset.get()` | Versioned CarDD retrieval |
| **1c** | **Feature Store** | **`Dataset.create()` child versioning** | **Upload new data version** |
| 2 | Orchestrated Experiment | `Task.init()` + `OutputModel` | Pre-trained weights registered |
| **2b** | **Orchestrated Experiment** | **`Task.init(training)` + `model.train()`** | **Fine-tune on new dataset** |
| 3 | Model Registry | Side-by-side comparison table | Best model tagged and promoted |
| 4 | Automated Pipeline | `PipelineDecorator` | Full CI/CD with quality gate |
| 5 | Performance Monitoring | `Task(monitor)` + `TriggerScheduler` | Auto-retrain on confidence drift |

### Updating with New Data — Quick Reference
```
1. Prepare updated CarDD_YOLO folder locally
2. Stage 1c  ->  set NEW_DATA_PATH  ->  run 3 cells  ->  note UPDATED_DATASET_ID
3. Stage 2b  ->  set model_name + epochs  ->  run 4 cells  ->  note TRAINED_TASK_ID
4. Stage 3   ->  add TRAINED_TASK_ID to MODELS_TO_COMPARE  ->  run comparison
5. Stage 4   ->  run_pipeline() auto-uses UPDATED_DATASET_ID
```

**Dashboard:** https://app.clear.ml -> Project: `Intelligent Vehicle Damage Assessment`

| Model | Weights File | Dataset |
|-------|-------------|----------|
| YOLO11m | `models/yolo11m_best.pt` | CarDD (6 damage classes) |
| YOLOv8m | `models/yolov8m_best.pt` | CarDD (6 damage classes) |
